### Task 4: Structured Output Parsers in LangChain


Imports and Define Model

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv


# Load environment variables
load_dotenv()

# Define a Pydantic model for the answer 
class AnswerWithCitation(BaseModel):
    
    answer: str = Field(description='The answer to the question')
    source: str = Field(description='The source document or section used')
    confidence: float = Field(description='Confidence score between 0 and 1')
    
print('Model defined successfully.')

Model defined successfully.


Create Parser and Format Instructions

In [5]:
# Create the parser from the Pydantic model
parser = PydanticOutputParser(pydantic_object=AnswerWithCitation)

# Get format instructions
format_instructions = parser.get_format_instructions()


print(format_instructions[:500])

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"description": "The an


Build Prompt and Chain

In [7]:
# Create the chat model
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Build prompt with format instructions
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. Answer the user question using the provided format.'),
    ('human', 'Question: {question}\n\n{format_instructions}')
])

# Partially fill the prompt with format instructions
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# Build the chain: prompt -> model -> parser
chain = prompt | llm | parser

print('Chain created successfully.')

Chain created successfully.


Invoke the Chain

In [8]:
# Invoke with a question
response = chain.invoke({'question': 'What is RAG?'})

print('Result')
print(response)
print(f'Type: {type(response)}')

Result
answer='RAG stands for Retrieval-Augmented Generation, a framework that combines retrieval of information from external sources with generative models to produce more accurate and contextually relevant responses.' source='General knowledge about AI frameworks and models.' confidence=0.9
Type: <class '__main__.AnswerWithCitation'>


Access individual Fields

In [9]:
print(f'Answer: {response.answer}')
print(f'Source: {response.source}')
print(f'Confidence: {response.confidence}')

Answer: RAG stands for Retrieval-Augmented Generation, a framework that combines retrieval of information from external sources with generative models to produce more accurate and contextually relevant responses.
Source: General knowledge about AI frameworks and models.
Confidence: 0.9


In [10]:
print('\nAs dictionary:', response.model_dump())


As dictionary: {'answer': 'RAG stands for Retrieval-Augmented Generation, a framework that combines retrieval of information from external sources with generative models to produce more accurate and contextually relevant responses.', 'source': 'General knowledge about AI frameworks and models.', 'confidence': 0.9}
